# Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import pandas as pd
from dotenv import load_dotenv
from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

sys.path.append("..")
load_dotenv("../../.env")

from pneuma_seeker.core.materializer.operation.semantic_joiner import SemanticJoiner
from pneuma_seeker.model.interface.model_factory import get_embed_model
from pneuma_seeker.model.interface.impl.azure import AzureOpenAILLM
from pneuma_seeker.core.materializer.operation.semantic_joiner import SyntacticSimMetric

In [ ]:
llm_path = "o4-mini"
embed_model_path = "model/weight/bge-base"

llm = AzureOpenAILLM(llm_path)
embed_model = get_embed_model()(embed_model_path)

In [ ]:
left_df = pd.read_csv("fisher_scientific_cy2023_price_roll_working_cap_file.csv")
right_df = pd.read_csv("fisher_q2_cy2025_quarterly_price_file_for_gordon_storeroom_copy_of_20250327_uochicago_orac.csv")

In [ ]:
semantic_joiner = SemanticJoiner(llm, embed_model)
res = semantic_joiner.semantic_join(
    left_df,
    right_df,
    ["part_number", "part_description"],
    ["part_id", "item_description"],
    syntactic_sim_metric=SyntacticSimMetric.JACCARD_QGRAM,
    top_k=2,
)
res

# Sample Data

In [ ]:
# import numpy as np
# from pyxdameraulevenshtein import damerau_levenshtein_distance
# from typing import Callable

# def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
#     """Cosine similarity in [0,1]."""
#     if np.all(a == 0) or np.all(b == 0):
#         return 0.0
#     a = a / np.linalg.norm(a)
#     b = b / np.linalg.norm(b)
#     return max(0.0, float(np.dot(a, b)))

# def edit_similarity(a: str, b: str) -> float:
#     """Normalized Damerau-Levenshtein similarity in [0,1]."""
#     if not a and not b:
#         return 1.0
#     max_len = max(len(a), len(b))
#     d = float(damerau_levenshtein_distance(a, b))
#     return max(0.0, min(1.0, 1.0 - (d / max_len)))

# def qgrams(s: str, q: int = 3, pad: bool = False):
#     if pad:
#         s = '^'*(q-1) + s + '$'*(q-1)
#     if len(s) < q:
#         return [s]
#     return [s[i:i+q] for i in range(len(s)-q+1)]

# def jaccard_qgram(a: str, b: str, q: int = 3, pad: bool = False):
#     A = set(qgrams(a, q, pad))
#     B = set(qgrams(b, q, pad))
#     if not A and not B:
#         return 1.0
#     return len(A & B) / len(A | B)

# def cosine_qgram(a: str, b: str, q: int = 3, pad: bool = False):
#     A = Counter(qgrams(a, q, pad))
#     B = Counter(qgrams(b, q, pad))
#     shared = set(A.keys()) & set(B.keys())
#     dot = sum(A[k]*B[k] for k in shared)
#     na = np.sqrt(sum(v*v for v in A.values()))
#     nb = np.sqrt(sum(v*v for v in B.values()))
#     if na == 0 or nb == 0:
#         return 0.0
#     return dot / (na*nb)

# class MinHash:
#     def __init__(self, num_perm: int = 128):
#         self.num_perm = num_perm
#         self.salts = [str(i).encode("utf8") for i in range(num_perm)]

#     @staticmethod
#     def _hash_bytes(b: bytes):
#         return int(hashlib.sha256(b).hexdigest(), 16) & ((1<<64)-1)

#     def signature(self, tokens):
#         token_bytes = [t.encode("utf8") for t in tokens]
#         sig = []
#         for salt in self.salts:
#             minv = (1<<64) - 1
#             for tb in token_bytes:
#                 h = self._hash_bytes(salt + b"|" + tb)
#                 if h < minv:
#                     minv = h
#             sig.append(minv)
#         return sig

# def minhash_jaccard_est(sig1, sig2):
#     equal = sum(1 for a, b in zip(sig1, sig2) if a == b)
#     return equal / len(sig1)

# def hybrid_similarity_cosine_edit_dist(
#     text_a: str,
#     text_b: str,
#     embed_func: Callable[[list[str]], np.ndarray],
#     alpha: float = 0.5,
# ) -> dict:
#     """
#     Compute embedding, edit, and hybrid similarity scores.
    
#     alpha = weight for embedding (cosine), (1-alpha) for edit distance.
#     """
#     # Get embeddings
#     emb_a, emb_b = embed_func([text_a, text_b])
#     cos = cosine_similarity(emb_a, emb_b)
#     edit = edit_similarity(text_a, text_b)
#     hybrid = alpha * cos + (1 - alpha) * edit
    
#     return {
#         "cosine": cos,
#         "edit": edit,
#         "hybrid": hybrid
#     }

# def hybrid_similarity_cosine_jaccard_qgram(
#     text_a: str, text_b: str, embed_func: Callable[[list[str]], np.ndarray], alpha: float = 0.5
# ):
#     emb_a, emb_b = embed_func([text_a, text_b])
#     cos = cosine_similarity(emb_a, emb_b)
#     jacc = jaccard_qgram(text_a, text_b)
#     hybrid = alpha * cos + (1 - alpha) * jacc
#     return {"cosine": cos, "jaccard_qgram": jacc, "hybrid": hybrid}


# def hybrid_similarity_cosine_minhash(
#     text_a: str, text_b: str, embed_func: Callable[[list[str]], np.ndarray], alpha: float = 0.5
# ):
#     emb_a, emb_b = embed_func([text_a, text_b])
#     cos = cosine_similarity(emb_a, emb_b)
#     mh = MinHash(num_perm=64)
#     sig_a = mh.signature(qgrams(text_a))
#     sig_b = mh.signature(qgrams(text_b))
#     mh_sim = minhash_jaccard_est(sig_a, sig_b)
#     hybrid = alpha * cos + (1 - alpha) * mh_sim
#     return {"cosine": cos, "minhash": mh_sim, "hybrid": hybrid}


# # Dummy embedder: convert string to vector of character ord values
# def get_embed(texts: list[str]):
#     return embed_model.encode(texts)

# # Test
# print(hybrid_similarity_cosine_edit_dist("kitten", "sitten", get_embed, alpha=0.7))
# print(hybrid_similarity_cosine_edit_dist("apple", "appl", get_embed, alpha=0.7))
# print(hybrid_similarity_cosine_edit_dist("apple", "banana", get_embed, alpha=0.7))